In [11]:
import os
import sys

# Sube de nivel el directorio para poder importar el módulo config_conexion
sys.path.append(os.path.abspath(".."))

import pandas as pd
from config_conexion import obtener_engine
from sqlalchemy import text

# 1. Conexión a la base de datos
engine = obtener_engine()

# 2. Lectura de la consulta T-SQL desde el archivo
with open("../sql/conjunto_trabajo.sql", "r", encoding="utf-8") as f:
    query_sql = text(f.read())

# 3. Parámetros de filtrado
parametros = {"fecha_inicio": "2005-01-01", "fecha_fin": "2006-12-31"}

# 4. Extracción al DataFrame con pandas.read_sql
df_trabajo = pd.read_sql(query_sql, con=engine, params=parametros)

# Confirmación de carga
print(
    f"Conjunto de trabajo cargado: {df_trabajo.shape[0]} filas y"
    f" {df_trabajo.shape} columnas."
)

# Muestra el DataFrame resultante
df_trabajo.head()

Conjunto de trabajo cargado: 41 filas y (41, 4) columnas.


,fecha,cantidad_alquileres,ingresos_totales,clientes_activos
0,2005-05-24,8,29.92,8
1,2005-05-25,137,573.63,129
2,2005-05-26,174,754.26,155
3,2005-05-27,166,684.34,150
4,2005-05-28,196,804.04,166


### Justificación del Reparto de Responsabilidades (SQL Server y Pandas)

Considerando lo visto en clase y la bibliografía oficial de la U5: 

Tomamos la eficiencia en manejo de grandes volúmenes de datos de SQL Server aprovechando su gran capacidad de optimización, permitiéndonos así, delegarle las tareas de: 

** Unión relacional (LEFT JOIN), para las tablas de interés (alquiler y pago) por medio de sus claves primarias y foráneas. 
** Filtrado (WHERE), delimitándole el rango temporal pasado por parámetros, manteniendo la seguridad y evitando inyecciones. 
** Agregación como GROUP BY y funciones de agregación, minimizando y reduciendo el volumen de datos transferidos. 

Una vez procesada la consulta en la BD, nos apoyamos en la flexibilidad de Pandas el consumo y análisis de datos, al momento de para trabajar con diversas fuentes, como la configuración de conexión con la base y el consumo de scripts TSQL. 

Así también el trabajo de seguridad en conjunto, realizando la otra labor de parámetros, para evitar inyecciones. 
Se definieron placeholders (:fecha_inicio y :fecha_fin) como parámetros, que SQL espera. Desde Pandas se brindan los valores reales mediante 'params' de 'pd.read_sql()', evitando concatenación de cadenas y f-strings. 

En el mismo orden, tomamos su capacidad de transformación para leer datos y cargarlos a un DataFrame, permitiéndonos así, generar una visualización más limpia para un próximo análisis. 


In [2]:
import os
import sys

# Sube de nivel el directorio para poder importar el módulo config_conexion
sys.path.append(os.path.abspath(".."))

import time
import pandas as pd
from config_conexion import obtener_engine
from esquema import TABLAS
from sqlalchemy import text

engine = obtener_engine()

# Rango de fechas parametrizado
params = {"fecha_inicio": "2005-01-01", "fecha_fin": "2006-12-31"}

# ==============================================================================
# ENFOQUE A: Íntegramente en T-SQL (Agregación en el motor mediante archivo .sql)
# ==============================================================================
t0_tsql = time.perf_counter()

# Leemos la consulta T-SQL desde el archivo externo
with open("../sql/conjunto_trabajo.sql", "r", encoding="utf-8") as f:
    query_tsql = text(f.read())

df_tsql = pd.read_sql(query_tsql, con=engine, params=params)
tiempo_tsql = time.perf_counter() - t0_tsql

# ==============================================================================
# ENFOQUE B: Íntegramente en pandas sobre datos crudos
# ==============================================================================
t0_pandas = time.perf_counter()

# 1. Extracción de tablas crudas sin agrupar ni filtrar
df_alquiler_crudo = pd.read_sql(
    text(f"SELECT id_alquiler, fecha_alquiler, id_cliente FROM {TABLAS['rental']}"),
    con=engine,
)
df_pago_crudo = pd.read_sql(
    text(f"SELECT id_alquiler, monto FROM {TABLAS['payment']}"), con=engine
)

# 2. Transformación y filtrado previo en pandas (Optimización)
df_alquiler_crudo["fecha_alquiler"] = pd.to_datetime(
    df_alquiler_crudo["fecha_alquiler"]
)
df_alquiler_filtrado = df_alquiler_crudo[
    (df_alquiler_crudo["fecha_alquiler"] >= params["fecha_inicio"])
    & (df_alquiler_crudo["fecha_alquiler"] <= params["fecha_fin"])
].copy()

df_alquiler_filtrado["fecha"] = df_alquiler_filtrado["fecha_alquiler"].dt.date

# 3. Réplica del LEFT JOIN mediante merge
df_merged = df_alquiler_filtrado.merge(df_pago_crudo, on="id_alquiler", how="left")

# 4. Agregación vectorizada en pandas (3 métricas)
df_pandas = (
    df_merged.groupby("fecha", as_index=False)
    .agg(
        cantidad_alquileres=("id_alquiler", "count"),
        ingresos_totales=("monto", lambda x: x.sum(min_count=1)),
        clientes_activos=("id_cliente", "nunique"),
    )
    .sort_values("fecha")
    .reset_index(drop=True)
)

# Rellenar posibles nulos de ingresos con 0.0 (COALESCE)
df_pandas["ingresos_totales"] = df_pandas["ingresos_totales"].fillna(0.0)

tiempo_pandas = time.perf_counter() - t0_pandas

# ==============================================================================
# IMPRESIÓN DE TIEMPOS
# ==============================================================================
print(f" Tiempo Enfoque T-SQL (Agregado): {tiempo_tsql:.4f} segundos")
print(f" Tiempo Enfoque pandas (Crudo):   {tiempo_pandas:.4f} segundos")
print(
    f" Diferencia de rendimiento:       El Enfoque T-SQL fue"
    f" {tiempo_pandas / tiempo_tsql:.2f}x más rápido.\n"
)

# ==============================================================================
# VERIFICACIÓN DE COINCIDENCIA ESTRICTA
# ==============================================================================
df_tsql["fecha"] = pd.to_datetime(df_tsql["fecha"])
df_pandas["fecha"] = pd.to_datetime(df_pandas["fecha"])
df_tsql["ingresos_totales"] = df_tsql["ingresos_totales"].astype(float)
df_pandas["ingresos_totales"] = df_pandas["ingresos_totales"].astype(float)

try:
    pd.testing.assert_frame_equal(
        df_tsql, df_pandas, check_dtype=False, check_exact=False, rtol=1e-5
    )
    print(
        "¡Éxito! Ambos resultados coinciden exactamente fila por fila y"
        " columna por columna."
    )
except AssertionError as e:
    print("Se detectaron diferencias entre ambos enfoques:")
    print(e)

 Tiempo Enfoque T-SQL (Agregado): 0.0876 segundos
 Tiempo Enfoque pandas (Crudo):   0.2620 segundos
 Diferencia de rendimiento:       El Enfoque T-SQL fue 2.99x más rápido.

¡Éxito! Ambos resultados coinciden exactamente fila por fila y columna por columna.
